# **Data Cleaning Notebook**
---
*TASKS:*
- Fixing the `TotalCharges` column because it needs to be numeric but it loads as a string
- Dropping the `customerID` column - not a feature really
- Checking and handling any nulls
- Encoding binary columns (from Yes/No to 1/0 respectively) for categorical columns.
- One-hot encoding multi-category columns.
---
## **Dataset overview**
- Source: IBM Telco Customer Churn dataset (Kaggle)
- Original shape: 7,043 rows × 21 columns
- Cleaned shape: 7,043 rows × 30 columns (extra columns from one-hot encoding)

## **Changes made**

### *1. TotalCharges: converted from str to float*
- it was loaded as `object` (string) because some entries were blank spaces
- so I converted to `float64` which created 11 nulls (new customers with no charge history yet)
- and then filled those 11 nulls with `0`

### *2. Dropped customerID*
- this column had a unique string identifier per customer e.g. `7590-VHVEG` and was dropped because it is not a feature so my model learning from customer IDs would be memorising, not learning.

### *3. Target column : Churn*
- Initially: `Yes` / `No` (str) became: `1` / `0` (int) respectively.
- Distribution: 1,869 churned (26.5%) vs 5,174 stayed (73.5%)

### *4. Binary Yes/No columns mapped to 1/0*
Columns affected: `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`
- Initially:`Yes` / `No` became: `1` / `0` respectively.

### *5. Gender mapped to 1/0*
- Initially: `Male` / `Female` became: `1` / `0` respectively.

### *6. Multi-category columns were one-hot encoded*
Original columns (10) replaced with dummy columns:
- `MultipleLines` → `MultipleLines_No phone service`, `MultipleLines_Yes`
- `InternetService` → `InternetService_Fiber optic`, `InternetService_No`
- `OnlineSecurity` → `OnlineSecurity_No internet service`, `OnlineSecurity_Yes`
- `OnlineBackup` → `OnlineBackup_No internet service`, `OnlineBackup_Yes`
- `DeviceProtection` → `DeviceProtection_No internet service`, `DeviceProtection_Yes`
- `TechSupport` → `TechSupport_No internet service`, `TechSupport_Yes`
- `StreamingTV` → `StreamingTV_No internet service`, `StreamingTV_Yes`
- `StreamingMovies` → `StreamingMovies_No internet service`, `StreamingMovies_Yes`
- `Contract` → `Contract_One year`, `Contract_Two year`
- `PaymentMethod` → `PaymentMethod_Credit card (automatic)`, `PaymentMethod_Electronic check`, `PaymentMethod_Mailed check`

`drop_first=True` was used as this drops one dummy per group to avoid multicollinearity
(e.g. if Contract_One year = 0 and Contract_Two year = 0, we already know it's Month-to-month)

---

## **Before vs after**

| Property | Before | After |
|---|---|---|
| Rows | 7,043 | 7,043 |
| Columns | 21 | 30 |
| Nulls | 11 (TotalCharges) | 0 |
| String columns | 17 | 0 |
| Numeric columns | 4 | 30 |
| Target encoding | Yes/No | 1/0 |

---

## **Output**
Saved to: `data/telco_churn_cleaned.csv`

### **Importing necessary libraries**

In [ ]:
import pandas as pd

### **Loading data**

In [ ]:
df = pd.read_csv("../data/telco_churn.csv")
# here, we see what we are working with

df.head()

### **Converting `TotalCharges` column to numeric**

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(df['TotalCharges'].dtype)

#### *null handling*

In [ ]:
# checking what nulls were created if any
print ("-" * 50)
print(df.isnull().sum())
# for any null that appears, it represents new customers with no charges yet

In [ ]:
# handling the nulls
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(df.isnull().sum())

### **Dropping `customerID`**

In [ ]:
df = df.drop(columns=['customerID'])
df.shape

### **Encoding the Target Column Churn**
*From 'Yes/No' to '1/0'*

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Churn'].value_counts()

### **Performing Binary encoding on categorical columns**

In [ ]:
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df[binary_cols].head()

#### *encoding the gender column to binary as well*

In [ ]:
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
df.head()

### **One-Hot encoding multi-category columns**

In [ ]:
multi_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'
]

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

df.shape

### **Final check**
*If we see all int64 or float64 dtypes and zero nulls, it means our data is clean and ready for modelling!*

In [ ]:
print(df.dtypes)
print(df.isnull().sum().sum())

df.head()

### *Saving Cleaned Data*

In [ ]:
df.to_csv('../data/telco_churn_cleaned.csv', index=False)

print("Saved cleaned data!")

### **Reflection: Why Encoding?**
- Machine Learning algorithms can't understand that "sunny" is different from "rainy".
- Encoding comes in to translate these categories into a language that machines can understand and work with.
- Incorrect encoding can introduce unintended biases or relationships

---
*reminder*

**Categorical Data Types**
1. **Nominal**: no inherent order categories. there's no natural ranking between weather condition for example.
2. **Ordinal**: have a meaningful order. Example: `Temperature` (Low, High, Very High) is ordinal. There's a *clear progression* from coldest to hottest.


**Methods of Encoding**
1. *Label Encoding*: assigns a unique integer to each category in a categorical variable.
    - Commonly used for ordinal variables where there's a clear oder to the categories, such as education levels.
2. *One-Hot Encoding*: creates a new binary column for each category in a categorical variable.
    - Used for nominal variables.
    - Useful when dealing with variables with a relatively small number of categories
3. *Binary Encoding*: represents each category as a binary num (0 and 1).
    - Useful when three are only two categories, mostly in a yes-no situation. Results in a single binary column.
4. *Target Encoding*: replaces each category with the mean of the target variable for that category.
    - Useful when there's a likely relationship between the categorical variable and the target variable.
    - Useful for high-cardinality features in datasets with a reasonable number of rows.
5. *Ordinal Encoding*: assigns ordered integers to ordinal categories based on their inherent order.
    - Useful for ordinal variables obviously (lol) as it preserves the natural ordering of the categories.
6. *Cyclic Encoding / Transformation*: transforms cyclic categorical variable into two numerical features that preserve the variable's cyclical nature. 
    - Typically uses sine and cosine transformations to represent the cyclical pattern. An exmaple could be a `Month` column.
    - Used for categorical variables that have a natural cyclical order, such as days of the week, months of the year, or hours of the day.  